# Long Document

In [1]:
long_document = """
University Academic Handbook

Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.

Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.

Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.

Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.

Examination Rules:
Students must carry their hall ticket to the exam hall.
Electronic devices are not allowed during exams.
"""

# Chunking Functions

In [2]:
def fixed_size_chunking(text, chunk_size=40):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

In [3]:
def overlapping_chunking(text, chunk_size=40, overlap=10):
    words = text.split()
    chunks = []
    
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    
    return chunks

In [4]:
def section_aware_chunking(text):
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    return chunks

Generate chunks using all three methods

In [5]:
fixed_chunks = fixed_size_chunking(long_document, chunk_size=40)
overlap_chunks = overlapping_chunking(long_document, chunk_size=40, overlap=10)
section_chunks = section_aware_chunking(long_document)

In [6]:
print("Fixed-size chunks:\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"Chunk {i}: {chunk}")
    print("-" * 60)

print("\nOverlapping chunks:\n")
for i, chunk in enumerate(overlap_chunks):
    print(f"Chunk {i}: {chunk}")
    print("-" * 60)

print("\nSection-aware chunks:\n")
for i, chunk in enumerate(section_chunks):
    print(f"Chunk {i}: {chunk}")
    print("-" * 60)

Fixed-size chunks:

Chunk 0: University Academic Handbook Attendance Policy: Students must maintain at least 75% attendance in every subject to be eligible for semester examinations. Students with attendance below 75% may not be allowed to write final exams unless special approval is granted. Hostel
------------------------------------------------------------
Chunk 1: Rules: Hostel gates close at 9 PM on weekdays and 10 PM on weekends. Students must carry their ID cards while entering the hostel. Fee Refund Policy: The refund deadline for semester fees is 15 days after the start of
------------------------------------------------------------
Chunk 2: classes. No refund will be issued after this deadline except under extraordinary circumstances. Library Policy: Students can borrow up to 4 books at a time. Books must be returned within 14 days to avoid a fine. Examination Rules: Students must
------------------------------------------------------------
Chunk 3: carry their hall ticket to

# Reusable retrieval function

In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

def build_faiss_index(chunks, model):
    embeddings = model.encode(chunks)
    embeddings = np.array(embeddings, dtype="float32")
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    
    return index, embeddings

c:\Tanisha\College\RAG-from-scratch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4946.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Function to retrieve top chunk

In [8]:
def retrieve_top_k(query, chunks, index, model, k=1):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding, dtype="float32")
    
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for rank, i in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk": chunks[i],
            "distance": distances[0][rank - 1]
        })
    
    return results

# Build three FAISS indexes

In [9]:
fixed_index, _ = build_faiss_index(fixed_chunks, model)
overlap_index, _ = build_faiss_index(overlap_chunks, model)
section_index, _ = build_faiss_index(section_chunks, model)

# Query

In [10]:
queries = [
    "What attendance is required for semester exams?",
    "When do hostel gates close?",
    "What is the refund deadline for fees?",
    "How many books can students borrow?",
    "Are electronic devices allowed in exams?"
]

# Results

In [11]:
for query in queries:
    print(f"\n{'='*80}")
    print("QUERY:", query)
    print(f"{'='*80}")
    
    print("\nFixed-size chunking result:")
    fixed_results = retrieve_top_k(query, fixed_chunks, fixed_index, model, k=1)
    print(fixed_results[0]["chunk"])
    print("Distance:", round(fixed_results[0]["distance"], 4))
    
    print("\nOverlapping chunking result:")
    overlap_results = retrieve_top_k(query, overlap_chunks, overlap_index, model, k=1)
    print(overlap_results[0]["chunk"])
    print("Distance:", round(overlap_results[0]["distance"], 4))
    
    print("\nSection-aware chunking result:")
    section_results = retrieve_top_k(query, section_chunks, section_index, model, k=1)
    print(section_results[0]["chunk"])
    print("Distance:", round(section_results[0]["distance"], 4))


QUERY: What attendance is required for semester exams?

Fixed-size chunking result:
University Academic Handbook Attendance Policy: Students must maintain at least 75% attendance in every subject to be eligible for semester examinations. Students with attendance below 75% may not be allowed to write final exams unless special approval is granted. Hostel
Distance: 0.4775

Overlapping chunking result:
University Academic Handbook Attendance Policy: Students must maintain at least 75% attendance in every subject to be eligible for semester examinations. Students with attendance below 75% may not be allowed to write final exams unless special approval is granted. Hostel
Distance: 0.4775

Section-aware chunking result:
Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.
Distance: 0.5294

QUERY: When do hostel